# Веб-інтерфейс — Результати аналізу

Цей ноутбук відображає результати всіх аналітичних модулів:
- Результати перевірки якості даних
- Результати дослідження даних
- Графіки візуалізації

In [ ]:
import os
import json
import glob
from IPython.display import display, HTML, Image, Markdown
import ipywidgets as widgets

---
## Опис проєкту

Контейнеризована аналітична система для обробки вихідної кореспонденції Держкомтелерадіо за грудень 2025 року.

**Архітектура**: `data_load` → `data_quality_analysis` / `data_research` / `visualization` → `web`

---
## 1. Результати перевірки якості даних

In [ ]:
quality_report_path = '/shared/reports/data_quality_report.json'

if os.path.exists(quality_report_path):
    with open(quality_report_path, 'r', encoding='utf-8') as f:
        quality_report = json.load(f)

    display(Markdown('### Загальна інформація'))
    display(HTML(f"""
    <table style='border-collapse:collapse;width:60%'>
    <tr><td style='border:1px solid #ddd;padding:8px'><b>Рядків</b></td><td style='border:1px solid #ddd;padding:8px'>{quality_report.get('total_rows','N/A')}</td></tr>
    <tr><td style='border:1px solid #ddd;padding:8px'><b>Стовпців</b></td><td style='border:1px solid #ddd;padding:8px'>{quality_report.get('total_columns','N/A')}</td></tr>
    <tr><td style='border:1px solid #ddd;padding:8px'><b>Памʼять</b></td><td style='border:1px solid #ddd;padding:8px'>{quality_report.get('memory_usage_kb','N/A')} KB</td></tr>
    <tr><td style='border:1px solid #ddd;padding:8px'><b>Дублікати</b></td><td style='border:1px solid #ddd;padding:8px'>{quality_report.get('total_duplicates','N/A')}</td></tr>
    </table>
    """))

    display(Markdown('### Пропущені значення'))
    missing = quality_report.get('missing_values', {})
    missing_pct = quality_report.get('missing_percent', {})
    rows_html = ''
    for col in missing:
        rows_html += f"<tr><td style='border:1px solid #ddd;padding:8px'>{col}</td><td style='border:1px solid #ddd;padding:8px'>{missing[col]}</td><td style='border:1px solid #ddd;padding:8px'>{missing_pct.get(col,'N/A')}%</td></tr>"
    display(HTML(f"<table style='border-collapse:collapse;width:60%'><tr><th style='border:1px solid #ddd;padding:8px'>Стовпець</th><th style='border:1px solid #ddd;padding:8px'>Пропусків</th><th style='border:1px solid #ddd;padding:8px'>%</th></tr>{rows_html}</table>"))

    display(Markdown('### Типи даних'))
    dtypes = quality_report.get('data_types', {})
    dt_rows = ''.join([f"<tr><td style='border:1px solid #ddd;padding:8px'>{c}</td><td style='border:1px solid #ddd;padding:8px'>{t}</td></tr>" for c, t in dtypes.items()])
    display(HTML(f"<table style='border-collapse:collapse;width:60%'><tr><th style='border:1px solid #ddd;padding:8px'>Стовпець</th><th style='border:1px solid #ddd;padding:8px'>Тип</th></tr>{dt_rows}</table>"))
else:
    display(HTML('<p style="color:red">⚠️ Звіт перевірки якості даних не знайдено.</p>'))

---
## 2. Результати дослідження даних

In [ ]:
research_report_path = '/shared/reports/data_research_report.json'

if os.path.exists(research_report_path):
    with open(research_report_path, 'r', encoding='utf-8') as f:
        research_report = json.load(f)

    display(Markdown('### Загальна статистика'))
    display(HTML(f"""
    <table style='border-collapse:collapse;width:60%'>
    <tr><td style='border:1px solid #ddd;padding:8px'><b>Кількість документів</b></td><td style='border:1px solid #ddd;padding:8px'>{research_report.get('total_records','N/A')}</td></tr>
    <tr><td style='border:1px solid #ddd;padding:8px'><b>Унікальних отримувачів</b></td><td style='border:1px solid #ddd;padding:8px'>{research_report.get('unique_recipients','N/A')}</td></tr>
    </table>
    """))

    top = research_report.get('top_recipients', {})
    if top:
        display(Markdown('### Топ-10 отримувачів'))
        tr_rows = ''.join([f"<tr><td style='border:1px solid #ddd;padding:8px'>{n}</td><td style='border:1px solid #ddd;padding:8px'>{c}</td></tr>" for n, c in top.items()])
        display(HTML(f"<table style='border-collapse:collapse;width:80%'><tr><th style='border:1px solid #ddd;padding:8px'>Отримувач</th><th style='border:1px solid #ddd;padding:8px'>К-ть</th></tr>{tr_rows}</table>"))

    clustering = research_report.get('clustering', {})
    if clustering:
        display(Markdown('### Кластеризація документів'))
        dist = clustering.get('distribution', {})
        kw = clustering.get('keywords', {})
        cl_rows = ''
        for cid in sorted(dist.keys()):
            keywords = ', '.join(kw.get(cid, []))
            cl_rows += f"<tr><td style='border:1px solid #ddd;padding:8px'>{cid}</td><td style='border:1px solid #ddd;padding:8px'>{dist[cid]}</td><td style='border:1px solid #ddd;padding:8px'>{keywords}</td></tr>"
        display(HTML(f"<table style='border-collapse:collapse;width:80%'><tr><th style='border:1px solid #ddd;padding:8px'>Кластер</th><th style='border:1px solid #ddd;padding:8px'>Документів</th><th style='border:1px solid #ddd;padding:8px'>Ключові слова</th></tr>{cl_rows}</table>"))

    date_stats = research_report.get('date_statistics', {})
    if date_stats:
        display(Markdown('### Статистика по датах'))
        display(HTML(f"""
        <table style='border-collapse:collapse;width:60%'>
        <tr><td style='border:1px solid #ddd;padding:8px'><b>Період</b></td><td style='border:1px solid #ddd;padding:8px'>{date_stats.get('min_date','')} — {date_stats.get('max_date','')}</td></tr>
        <tr><td style='border:1px solid #ddd;padding:8px'><b>Сер. документів/день</b></td><td style='border:1px solid #ddd;padding:8px'>{date_stats.get('mean_docs_per_day','N/A')}</td></tr>
        <tr><td style='border:1px solid #ddd;padding:8px'><b>Максимум/день</b></td><td style='border:1px solid #ddd;padding:8px'>{date_stats.get('max_docs_per_day','N/A')}</td></tr>
        </table>
        """))
else:
    display(HTML('<p style="color:red">⚠️ Звіт дослідження даних не знайдено.</p>'))

---
## 3. Візуалізації

In [ ]:
plots_dir = '/shared/plots'
plot_files = sorted(glob.glob(os.path.join(plots_dir, '*.png')))

if plot_files:
    for plot_path in plot_files:
        fname = os.path.basename(plot_path)
        display(Markdown(f'### {fname}'))
        display(Image(filename=plot_path, width=800))
else:
    display(HTML('<p style="color:red">⚠️ Графіки не знайдено. Переконайтесь, що сервіс visualization відпрацював.</p>'))

---
*Запуск системи: `docker compose up --build` | Веб-інтерфейс: http://localhost:8866*